# Full TEM Column — Interactive

Load a previously exported `TEMModel` and explore brightness (condenser CL3)
and magnification (IL1–IL3 + projector) interactively.

The hex DAC codes shown mirror what the JEOL console would display.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from temgym_core.tem_model import load_tem_model_json, dac_int_to_hex
from temgym_core.source import make_waist_divergence_rays
from temgym_core.plotting import plot_model, PlotParams, legacy_beam_plot_params

## 1. Load the model

In [ ]:
model_path = Path("full_tem_model.json")
if not model_path.exists():
    # Try finding the model in the examples directory
    model_path = Path("examples/microscope_models/full_tem_model.json")
model = load_tem_model_json(model_path)
print(f"Loaded model: {model.model_type}, {model.voltage_v/1e3:.0f} kV")
print(f"Illumination lenses: {model.illumination.geometry.lens_names}")
print(f"Projection  lenses:  {model.projection.geometry.lens_names}")

## 2. Function API

Use `realize_full` to query DAC hex codes and focal lengths at any
brightness / magnification setting.

In [ ]:
def show_state(brightness_nm: float = 250.0, magnification: float = 100_000):
    """Print lens state and DAC hex for both sub-systems."""
    state = model.realize_full(brightness_nm=brightness_nm, magnification=magnification)

    print(f"\n{'='*60}")
    print(f"  Brightness (FWHM): {brightness_nm:.0f} nm")
    print(f"  Magnification:     {magnification:,.0f}×")
    print(f"{'='*60}")

    print("\n  ILLUMINATION")
    illum = state['illumination']
    for name, hexval in illum['dac_hex'].items():
        idx = model.illumination.geometry.lens_names.index(name)
        f = illum['focal_lengths_m'][idx]
        print(f"    {name:8s}  DAC={hexval}  f={f*1e3:8.3f} mm")

    print("\n  PROJECTION")
    proj = state['projection']
    for name, hexval in proj['dac_hex'].items():
        idx = model.projection.geometry.lens_names.index(name)
        f = proj['focal_lengths_m'][idx]
        print(f"    {name:8s}  DAC={hexval}  f={f*1e3:8.3f} mm")
    print()

show_state(250, 100_000)

## 3. Visual ray trace

Build the full component chain and plot with `plot_model`.

In [ ]:
def trace_and_plot(
    brightness_nm: float = 250.0,
    magnification: float = 100_000,
    waist_m: float = 1e-6,
    figsize: tuple = (6, 14),
):
    """Build, trace, and plot the full TEM column."""
    comps = model.build_full_column(
        brightness_nm=brightness_nm,
        magnification=magnification,
    )
    rays = make_waist_divergence_rays(
        waist=waist_m,
        voltage=model.voltage_v,
        z=0.0,
    )
    fig, ax = plot_model(
        comps,
        rays=rays,
        plot_params=legacy_beam_plot_params(),
    )
    ax.set_title(f"FWHM={brightness_nm:.0f} nm, Mag={magnification:,.0f}×")
    fig.set_size_inches(*figsize)
    fig.tight_layout()
    return fig, ax

trace_and_plot(250, 100_000)

## 4. Interactive widgets

Use ipywidgets sliders to adjust brightness and magnification in real time.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    brightness_slider = widgets.FloatSlider(
        value=250, min=50, max=500, step=10,
        description='FWHM (nm):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px'),
    )

    # Available magnification steps from the dial curve
    mag_options = model.projection.dial_curves['magnification'].control_values.astype(int).tolist()
    mag_dropdown = widgets.Dropdown(
        options=[(f"{m:,}×", m) for m in mag_options],
        value=100_000,
        description='Mag:',
        style={'description_width': 'initial'},
    )

    hex_output = widgets.Output(layout=widgets.Layout(width='400px'))
    plot_output = widgets.Output()

    def _update(change=None):
        fwhm = brightness_slider.value
        mag = mag_dropdown.value

        # Update hex display
        with hex_output:
            clear_output(wait=True)
            show_state(fwhm, mag)

        # Update plot
        with plot_output:
            clear_output(wait=True)
            trace_and_plot(fwhm, mag)
            plt.show()

    brightness_slider.observe(_update, names='value')
    mag_dropdown.observe(_update, names='value')

    controls = widgets.VBox([brightness_slider, mag_dropdown])
    display(widgets.HBox([widgets.VBox([controls, hex_output]), plot_output]))
    _update()  # initial render

except ImportError:
    print("ipywidgets not installed — skipping interactive controls.")
    print("Use the function API (show_state / trace_and_plot) directly.")